# Task 1 — Full Sensor and Model-Regime Comparison


This notebook reproduces the full Task 1 comparison table from the
review report across all seven sensor combinations:

1. classical models without elapsed time;
2. classical models with elapsed time;
3. deep sequence models without elapsed time.

The earlier focused Task 1 notebook reproduced only the optimized
`OPTI2_RELATIVE_ONLY` Transformer. This notebook instead reproduces
the complete seven-combination comparison used in the report.


## Output convention

The notebooks report two distinct result types:

- **Pooled metrics**, matching the historical report: all held-out
  predictions are combined before accuracy and macro-F1 are calculated.
- **Fold mean ± SD**, measuring variation across held-out groups.

The best configuration in each regime is selected by pooled macro-F1,
exactly as in the report. The fold mean and SD are then attached as
additional publication statistics.

With the default report-reproduction settings:

- Classical: 7 sensor combinations × 2 time conditions ×
  2 models × 2 k-values = **56 configurations**, each under LOGO.
- DL: 7 sensor combinations × 4 architectures × 1 seed =
  **28 LOGO runs**.

In [1]:
# Optional Google Drive mount for Google Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except (ImportError, ModuleNotFoundError):
    print(
        "Not running in Colab. Update DATA_ROOT in the "
        "configuration cell."
    )


Mounted at /content/drive


In [2]:
# ================================================================
# SETUP
# ================================================================

import os
import re
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", None)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cpu


In [3]:
# ================================================================
# CONFIG
# ================================================================

DATA_ROOT = "/content/drive/MyDrive/thesis/data"

# Original feature tables used for the report.
ACTIVITY_DATA_PATH = (
    f"{DATA_ROOT}/INTERACTION_ABLATIONS/"
    "activity3_advanced_merged_10s_features.csv"
)

INTERACTION_DATA_PATHS = [
    (
        f"{DATA_ROOT}/INTERACTION_BINARY_5S_SPECIALIZED_OE/"
        "binary_5s_specialized_oe_merged_all_features.csv"
    ),
    (
        f"{DATA_ROOT}/INTERACTION_BINARY_5S_ADVANCED_FEATURES/"
        "binary_5s_all_sensor_advanced_features.csv"
    ),
]

OUT_DIR = f"{DATA_ROOT}/PUBLICATION_TASK1_FULL_COMPARISON"
os.makedirs(OUT_DIR, exist_ok=True)

RUN_TASKS = ["interaction_vs_noninteraction"]

SENSOR_COMBINATIONS = {
    "OE": ["OE"],
    "OPTI": ["OPTI"],
    "XSENS": ["XSENS"],
    "OE_OPTI": ["OE", "OPTI"],
    "OE_XSENS": ["OE", "XSENS"],
    "OPTI_XSENS": ["OPTI", "XSENS"],
    "OE_OPTI_XSENS": ["OE", "OPTI", "XSENS"],
}

RUN_CLASSICAL = True
RUN_DL = True

# ----------------------------------------------------------------
# REPORT REPRODUCTION SETTINGS
# ----------------------------------------------------------------
# The report's full seven-combination classical table was produced
# from Logistic Regression and Linear SVC at k=80 and k=200.
REPORT_REPRODUCTION_MODE = True

# Set REPORT_REPRODUCTION_MODE=False to run the larger exploratory
# grid: RBF-SVC, Random Forest and Extra Trees, plus more k values.
K_CLASSICAL_FULL = [40, 80, 120, 200, "all"]
TIME_CONDITIONS = ["no_elapsed", "with_elapsed"]

# DL comparison: no elapsed time only.
# All four sequence architectures are screened because different
# models win for different tasks/sensor combinations in the report.
RUN_DL_MODEL_TYPES = ["lstm", "bilstm", "gru", "transformer"]
K_DL = [120]

# Seed 42 reproduces the historical report comparison.
# Adding seeds, e.g. [42, 7, 21], additionally enables seed-level SD.
SEEDS = [42]

FAST_MAX_EPOCHS = 25
FAST_PATIENCE = 4
FAST_BATCH_SIZE = 256
FAST_PRED_BATCH_SIZE = 1024

# Full leave-one-group-out evaluation.
MAX_LOGO_FOLDS = None

# Row-level predictions are not needed for mean/SD and create large files.
SAVE_CLASSICAL_PREDICTIONS = False
SAVE_FAST_DL_PREDICTIONS = False

RANDOM_STATE = 42


## Data and feature preparation

This section uses the same feature-detection and task-construction
logic as the original all-sensor ablation notebook. Elapsed time is
appended only to the classical `with_elapsed` condition. It is never
included in the DL comparison.

In [ ]:
# ================================================================
# GENERAL HELPERS
# ================================================================

def safe_name(x):
    x = str(x)
    x = re.sub(r"[^A-Za-z0-9_]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x


def unique_feats(feats):
    return list(dict.fromkeys(feats))


def find_first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None


def infer_window_seconds(df, start_col, end_col, default):
    if start_col in df.columns and end_col in df.columns:
        dur = pd.to_numeric(df[end_col], errors="coerce") - pd.to_numeric(df[start_col], errors="coerce")
        val = float(np.nanmedian(dur))
        if np.isfinite(val) and val > 0:
            return val
    return float(default)


def add_elapsed_min(df, group_col, start_col, end_col):
    df = df.copy()
    if "elapsed_min" in df.columns:
        df["elapsed_min"] = pd.to_numeric(df["elapsed_min"], errors="coerce")
        return df

    if start_col not in df.columns:
        raise ValueError(f"Cannot create elapsed_min because {start_col} is missing.")

    if end_col in df.columns:
        df["window_mid"] = (
            pd.to_numeric(df[start_col], errors="coerce") +
            pd.to_numeric(df[end_col], errors="coerce")
        ) / 2.0
    else:
        df["window_mid"] = pd.to_numeric(df[start_col], errors="coerce")

    df["elapsed_min"] = (df["window_mid"] - df.groupby(group_col)["window_mid"].transform("min")) / 60.0
    return df


BAD_TOKENS = [
    "label", "target", "class", "group", "window", "time", "elapsed",
    "pred", "prediction", "correct", "fold", "split", "index"
]


def looks_bad_feature_name(c):
    cl = str(c).lower()
    return any(tok in cl for tok in BAD_TOKENS)


def is_oe_feature(c):
    c = str(c)
    cl = c.lower()
    if looks_bad_feature_name(c):
        return False
    return (
        c.startswith("oe__")
        or c.startswith("oe_")
        or c.startswith("oebest__")
        or c.startswith("ear_")
        or c.startswith("mag__")
        or c.startswith("mag_")
    )


def is_opti_feature(c):
    c = str(c)
    cl = c.lower()
    if looks_bad_feature_name(c):
        return False

    # Current advanced merged datasets normally use opti2_ / opti2__ prefixes.
    if c.startswith("opti2_") or c.startswith("opti2__") or c.startswith("opti__") or c.startswith("opti_"):
        return True

    # Fallback for older unprefixed OptiTrack feature names.
    # Kept conservative to avoid catching OE/XSens features.
    fallback_tokens = [
        "dist_close", "dist_mid", "dist_far", "dist_disp",
        "centroid", "spread", "triangle", "perimeter", "compactness",
        "nearest", "farthest", "pairdist", "pair_dist",
        "approach", "separation", "proximity", "relative_pos",
    ]
    if any(tok in cl for tok in fallback_tokens):
        return True

    # Some old OptiTrack columns used simple speed names.
    if cl in {"speed_min", "speed_mid", "speed_max", "centroid_speed"}:
        return True

    return False


def is_xsens_feature(c):
    c = str(c)
    cl = c.lower()
    if looks_bad_feature_name(c):
        return False
    return (
        c.startswith("xsens2__")
        or c.startswith("xsens2_")
        or c.startswith("xsens__")
        or c.startswith("xsens_")
    )


def clean_feature_list(dataframe, feats, label_cols=None, include_elapsed=False):
    label_cols = set(label_cols or [])
    bad_cols = set(label_cols) | {
        "group", "window_start", "window_end", "window_mid", "video_time_s", "time_s",
        "label", "target", "class", "activity", "activity_class", "general_class",
        "binary_label", "recognition_label", "task_label", "pred", "prediction", "correct",
    }

    cleaned = []
    for f in unique_feats(feats):
        if f not in dataframe.columns:
            continue
        if f in bad_cols:
            continue

        fl = str(f).lower()
        if f == "elapsed_min":
            if include_elapsed:
                cleaned.append(f)
            continue

        if not include_elapsed and "elapsed" in fl:
            continue
        if any(tok in fl for tok in ["label", "target", "pred", "prediction", "correct"]):
            continue

        x = pd.to_numeric(dataframe[f], errors="coerce").values
        if np.isfinite(x).sum() < 20:
            continue
        if np.nanstd(x) < 1e-12:
            continue
        cleaned.append(f)

    return unique_feats(cleaned)


def make_classical_models(n_classes):
    return {
        "logreg_C1": LogisticRegression(
            C=1.0,
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear" if n_classes == 2 else "lbfgs",
            multi_class="auto",
            random_state=RANDOM_STATE,
        ),
        "linearSVC_C1": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=10000,
            dual=False,
        ),
        "rbfSVC_C1_gscale": SVC(
            C=1.0,
            gamma="scale",
            kernel="rbf",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "rf_leaf2": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "extraTrees_leaf1": ExtraTreesClassifier(
            n_estimators=500,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    }


def build_pipeline(model, k, n_features):
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
    if k != "all":
        actual_k = min(int(k), n_features)
        steps.append(("select", SelectKBest(f_classif, k=actual_k)))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def metric_dict(y_true, y_pred, label_order, prefix=""):
    out = {
        prefix + "accuracy": accuracy_score(y_true, y_pred),
        prefix + "macro_f1": f1_score(y_true, y_pred, labels=label_order, average="macro", zero_division=0),
        prefix + "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }

    pr, rc, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=label_order, zero_division=0
    )

    for lab, p, r, f, s in zip(label_order, pr, rc, f1, sup):
        safe = safe_name(lab)
        out[prefix + f"precision_{safe}"] = p
        out[prefix + f"recall_{safe}"] = r
        out[prefix + f"f1_{safe}"] = f
        out[prefix + f"support_{safe}"] = int(s)

    return out


In [ ]:
# ================================================================
# LOAD DATASETS AND CREATE TASKS
# ================================================================

# ---------- Load activity 10s dataset ----------
if not os.path.exists(ACTIVITY_DATA_PATH):
    raise FileNotFoundError(
        f"Could not find activity dataset:\n{ACTIVITY_DATA_PATH}\n"
        "Run the advanced merged 10s feature-generation cell first."
    )

activity_df_raw = pd.read_csv(ACTIVITY_DATA_PATH)
activity_df_raw["recognition_label"] = activity_df_raw["recognition_label"].astype(str).str.strip()
activity_df_raw = add_elapsed_min(activity_df_raw, "group", "window_start", "window_end")
activity_window_s = infer_window_seconds(activity_df_raw, "window_start", "window_end", default=10.0)

print("Loaded activity dataset:", ACTIVITY_DATA_PATH)
print("Shape:", activity_df_raw.shape)
print("Estimated window seconds:", activity_window_s)
display(activity_df_raw["recognition_label"].value_counts())

# ---------- Load binary interaction 5s dataset ----------
interaction_path = find_first_existing(INTERACTION_DATA_PATHS)
if interaction_path is None:
    print("WARNING: interaction binary dataset not found. Interaction task will be skipped.")
    interaction_df_raw = None
    interaction_window_s = 5.0
else:
    interaction_df_raw = pd.read_csv(interaction_path)
    interaction_df_raw["binary_label"] = interaction_df_raw["binary_label"].astype(str).str.strip()
    interaction_df_raw = add_elapsed_min(interaction_df_raw, "group", "window_start", "window_end")
    interaction_window_s = infer_window_seconds(interaction_df_raw, "window_start", "window_end", default=5.0)

    print("\nLoaded interaction dataset:", interaction_path)
    print("Shape:", interaction_df_raw.shape)
    print("Estimated window seconds:", interaction_window_s)
    display(interaction_df_raw["binary_label"].value_counts())


# ---------- Feature extraction per dataset ----------
def get_modality_features(df, label_cols=None):
    label_cols = label_cols or []

    oe_raw = [c for c in df.columns if is_oe_feature(c)]
    opti_raw = [c for c in df.columns if is_opti_feature(c)]
    xsens_raw = [c for c in df.columns if is_xsens_feature(c)]

    # Make modality sets mutually exclusive if broad fallback captured something unexpectedly.
    oe = clean_feature_list(df, oe_raw, label_cols=label_cols, include_elapsed=False)
    opti = clean_feature_list(df, [c for c in opti_raw if c not in oe], label_cols=label_cols, include_elapsed=False)
    xsens = clean_feature_list(df, [c for c in xsens_raw if c not in oe and c not in opti], label_cols=label_cols, include_elapsed=False)

    return {"OE": oe, "OPTI": opti, "XSENS": xsens}


def make_combo_feature_sets(df, modality_features, label_cols=None):
    combo_sets = {}
    combo_sets_elapsed = {}

    for combo_name, modalities in SENSOR_COMBINATIONS.items():
        feats = []
        for m in modalities:
            feats.extend(modality_features.get(m, []))
        feats = clean_feature_list(df, feats, label_cols=label_cols, include_elapsed=False)
        feats_elapsed = clean_feature_list(
            df,
            feats + (["elapsed_min"] if "elapsed_min" in df.columns else []),
            label_cols=label_cols,
            include_elapsed=True,
        )
        combo_sets[combo_name] = feats
        combo_sets_elapsed[combo_name] = feats_elapsed

    return combo_sets, combo_sets_elapsed


# ---------- Task preparation ----------
def prepare_task(task_name):
    if task_name == "interaction_vs_noninteraction":
        if interaction_df_raw is None:
            return None

        df = interaction_df_raw.copy()
        label_col = "binary_label"
        labels = ["interaction", "non_interaction"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [6, 12, 18]   # 5s windows -> 30/60/90s
        window_s = interaction_window_s

    elif task_name == "conversation_vs_nonconversation":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        core = ["co_building", "co_merging", "conversation"]
        df = df[df[label_col].isin(core)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = np.where(df[label_col] == "conversation", "conversation", "non_conversation")
        labels = ["conversation", "non_conversation"]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "conversation_vs_building":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["conversation", "co_building"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "conversation_vs_merging":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["conversation", "co_merging"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "merging_vs_building":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["co_merging", "co_building"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    elif task_name == "three_class_activity":
        df = activity_df_raw.copy()
        label_col = "recognition_label"
        labels = ["co_building", "co_merging", "conversation"]
        df = df[df[label_col].isin(labels)].copy().reset_index(drop=True)
        target_col = "task_label"
        df[target_col] = df[label_col]
        default_seq_lens = [3, 6, 9]
        window_s = activity_window_s

    else:
        raise ValueError(f"Unknown task: {task_name}")

    if len(df) == 0:
        print("Skipping empty task:", task_name)
        return None

    modality_features = get_modality_features(df, label_cols=[label_col, target_col])
    combo_sets, combo_sets_elapsed = make_combo_feature_sets(df, modality_features, label_cols=[label_col, target_col])

    spec = {
        "task_name": task_name,
        "df": df,
        "label_col": label_col,
        "target_col": target_col,
        "label_order": labels,
        "group_col": "group",
        "start_col": "window_start",
        "end_col": "window_end",
        "window_seconds": window_s,
        "seq_lens": default_seq_lens,
        "modality_features": modality_features,
        "combo_features": combo_sets,
        "combo_features_elapsed": combo_sets_elapsed,
        "out_dir": os.path.join(OUT_DIR, task_name),
    }
    os.makedirs(spec["out_dir"], exist_ok=True)
    return spec


task_specs = []
for task in RUN_TASKS:
    spec = prepare_task(task)
    if spec is not None:
        task_specs.append(spec)

print("\n" + "=" * 100)
print("PREPARED TASKS AND SENSOR COMBINATIONS")
print("=" * 100)

for spec in task_specs:
    print("\nTASK:", spec["task_name"])
    print("Rows:", len(spec["df"]))
    print("Classes:", spec["label_order"])
    print("Window seconds:", spec["window_seconds"])
    print("Seq lens:", spec["seq_lens"])
    print("Modality counts:", {k: len(v) for k, v in spec["modality_features"].items()})
    print("Combination counts, no elapsed:", {k: len(v) for k, v in spec["combo_features"].items()})
    print("Combination counts, with elapsed:", {k: len(v) for k, v in spec["combo_features_elapsed"].items()})
    display(spec["df"][spec["target_col"]].value_counts())
    display(pd.crosstab(spec["df"][spec["group_col"]], spec["df"][spec["target_col"]]))

    feature_count_rows = []
    for combo in SENSOR_COMBINATIONS:
        feature_count_rows.append({
            "task": spec["task_name"],
            "sensor_combo": combo,
            "n_no_elapsed": len(spec["combo_features"][combo]),
            "n_with_elapsed": len(spec["combo_features_elapsed"][combo]),
        })
    pd.DataFrame(feature_count_rows).to_csv(os.path.join(spec["out_dir"], f"{spec['task_name']}_feature_counts.csv"), index=False)


Loaded activity dataset: /content/drive/MyDrive/thesis/data/INTERACTION_ABLATIONS/activity3_advanced_merged_10s_features.csv
Shape: (992, 1543)
Estimated window seconds: 10.0


,count
recognition_label,
co_building,548
conversation,311
co_merging,133



Loaded interaction dataset: /content/drive/MyDrive/thesis/data/INTERACTION_BINARY_5S_SPECIALIZED_OE/binary_5s_specialized_oe_merged_all_features.csv
Shape: (4579, 1515)
Estimated window seconds: 5.0


,count
binary_label,
non_interaction,2304
interaction,2275



PREPARED TASKS AND SENSOR COMBINATIONS

TASK: interaction_vs_noninteraction
Rows: 4579
Classes: ['interaction', 'non_interaction']
Window seconds: 5.0
Seq lens: [6, 12, 18]
Modality counts: {'OE': 457, 'OPTI': 387, 'XSENS': 615}
Combination counts, no elapsed: {'OE': 457, 'OPTI': 387, 'XSENS': 615, 'OE_OPTI': 844, 'OE_XSENS': 1072, 'OPTI_XSENS': 1002, 'OE_OPTI_XSENS': 1459}
Combination counts, with elapsed: {'OE': 458, 'OPTI': 388, 'XSENS': 616, 'OE_OPTI': 845, 'OE_XSENS': 1073, 'OPTI_XSENS': 1003, 'OE_OPTI_XSENS': 1460}


,count
task_label,
non_interaction,2304
interaction,2275


task_label,interaction,non_interaction
group,,
1,213,153
2,299,267
3,205,330
5,391,401
6,152,122
7,332,247
8,244,220
9,361,352
10,78,212


## Classical comparison

Default report-reproduction mode evaluates Logistic Regression and
Linear SVC with `k=80` and `k=200`, matching the complete comparison
table in the report.

Set `REPORT_REPRODUCTION_MODE=False` in the configuration cell to
additionally evaluate RBF-SVC, Random Forest and Extra Trees over the
larger exploratory k-grid.

In [ ]:
# ================================================================
# CLASSICAL LOGO RUNNER WITH FOLD MEAN ± SD
# ================================================================

from sklearn.base import clone
from sklearn.model_selection import LeaveOneGroupOut

if REPORT_REPRODUCTION_MODE:
    CLASSICAL_MODEL_NAMES = ["logreg_C1", "linearSVC_C1"]
    CLASSICAL_K_VALUES = [80, 200]
else:
    CLASSICAL_MODEL_NAMES = [
        "logreg_C1",
        "linearSVC_C1",
        "rbfSVC_C1_gscale",
        "rf_leaf2",
        "extraTrees_leaf1",
    ]
    CLASSICAL_K_VALUES = K_CLASSICAL_FULL


def run_classical_for_task_and_sensor(spec, sensor_combo):
    task_name = spec["task_name"]
    df = spec["df"]

    y = df[spec["target_col"]].astype(str).values
    groups = df[spec["group_col"]].values
    label_order = spec["label_order"]
    n_classes = len(label_order)

    all_models = make_classical_models(n_classes)
    models = {
        name: all_models[name]
        for name in CLASSICAL_MODEL_NAMES
        if name in all_models
    }

    summary_rows = []
    fold_rows = []
    pred_rows = []

    logo = LeaveOneGroupOut()

    for time_condition in TIME_CONDITIONS:
        if time_condition == "no_elapsed":
            feats = spec["combo_features"][sensor_combo]
        else:
            feats = spec["combo_features_elapsed"][sensor_combo]

        if len(feats) == 0:
            print(
                f"Skipping {task_name} | {sensor_combo} | "
                f"{time_condition}: no usable features"
            )
            continue

        X = (
            df[feats]
            .apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
            .values
        )

        for model_name, model in models.items():
            for k in CLASSICAL_K_VALUES:
                if k != "all" and int(k) > len(feats):
                    continue

                print(
                    f"Classical | {task_name} | {sensor_combo} | "
                    f"{time_condition} | {model_name} | k={k}"
                )

                pipe = build_pipeline(model, k, len(feats))

                y_true_all = []
                y_pred_all = []
                group_all = []
                row_index_all = []

                for fold, (tr_idx, te_idx) in enumerate(
                    logo.split(X, y, groups), start=1
                ):
                    if len(np.unique(y[tr_idx])) < 2:
                        continue

                    pipe_fold = clone(pipe)
                    pipe_fold.fit(X[tr_idx], y[tr_idx])
                    pred = pipe_fold.predict(X[te_idx])

                    fold_metric = metric_dict(
                        y[te_idx], pred, label_order
                    )
                    fold_rows.append({
                        "task": task_name,
                        "sensor_combo": sensor_combo,
                        "time_condition": time_condition,
                        "model": model_name,
                        "k": k,
                        "n_features": len(feats),
                        "fold": fold,
                        "test_group": groups[te_idx][0],
                        "n_test_rows": len(te_idx),
                        "accuracy": fold_metric["accuracy"],
                        "macro_f1": fold_metric["macro_f1"],
                        "balanced_accuracy": fold_metric[
                            "balanced_accuracy"
                        ],
                    })

                    y_true_all.extend(y[te_idx])
                    y_pred_all.extend(pred)
                    group_all.extend(groups[te_idx])
                    row_index_all.extend(te_idx)

                if not y_true_all:
                    continue

                y_true_all = np.asarray(y_true_all)
                y_pred_all = np.asarray(y_pred_all)

                pooled = metric_dict(
                    y_true_all, y_pred_all, label_order
                )
                pooled.update({
                    "task": task_name,
                    "sensor_combo": sensor_combo,
                    "time_condition": time_condition,
                    "model": model_name,
                    "k": k,
                    "n_features": len(feats),
                    "n_rows_evaluated": len(y_true_all),
                })
                summary_rows.append(pooled)

                if SAVE_CLASSICAL_PREDICTIONS:
                    for idx, g, yt, yp in zip(
                        row_index_all,
                        group_all,
                        y_true_all,
                        y_pred_all,
                    ):
                        pred_rows.append({
                            "task": task_name,
                            "sensor_combo": sensor_combo,
                            "time_condition": time_condition,
                            "model": model_name,
                            "k": k,
                            "row_index": int(idx),
                            "group": g,
                            "y_true": yt,
                            "y_pred": yp,
                            "correct": bool(yt == yp),
                        })

    summary = pd.DataFrame(summary_rows)
    folds = pd.DataFrame(fold_rows)
    preds = pd.DataFrame(pred_rows)

    if summary.empty:
        return summary, folds, preds, pd.DataFrame()

    group_keys = [
        "task",
        "sensor_combo",
        "time_condition",
        "model",
        "k",
        "n_features",
    ]

    fold_stats = (
        folds.groupby(group_keys, as_index=False)
        .agg(
            fold_accuracy_mean=("accuracy", "mean"),
            fold_accuracy_std=("accuracy", "std"),
            fold_macro_f1_mean=("macro_f1", "mean"),
            fold_macro_f1_std=("macro_f1", "std"),
            fold_balanced_accuracy_mean=(
                "balanced_accuracy", "mean"
            ),
            fold_balanced_accuracy_std=(
                "balanced_accuracy", "std"
            ),
            n_folds=("test_group", "nunique"),
        )
    )

    summary = summary.merge(
        fold_stats,
        on=group_keys,
        how="left",
        validate="one_to_one",
    )

    summary = (
        summary.sort_values(
            ["macro_f1", "accuracy"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    # Match the report: select the best configuration by pooled macro-F1.
    best_per_condition = (
        summary.sort_values(
            ["time_condition", "macro_f1", "accuracy"],
            ascending=[True, False, False],
        )
        .groupby("time_condition", as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    out_dir = os.path.join(spec["out_dir"], sensor_combo)
    os.makedirs(out_dir, exist_ok=True)

    summary.to_csv(
        os.path.join(
            out_dir,
            f"{task_name}_{sensor_combo}_classical_summary_with_std.csv",
        ),
        index=False,
    )
    folds.to_csv(
        os.path.join(
            out_dir,
            f"{task_name}_{sensor_combo}_classical_fold_metrics.csv",
        ),
        index=False,
    )
    best_per_condition.to_csv(
        os.path.join(
            out_dir,
            f"{task_name}_{sensor_combo}_classical_best_per_condition_with_std.csv",
        ),
        index=False,
    )

    if SAVE_CLASSICAL_PREDICTIONS and not preds.empty:
        preds.to_csv(
            os.path.join(
                out_dir,
                f"{task_name}_{sensor_combo}_classical_predictions.csv",
            ),
            index=False,
        )

    display(best_per_condition.round(4))
    return summary, folds, preds, best_per_condition


all_classical_summaries = []
all_classical_folds = []
all_classical_best = []

if RUN_CLASSICAL:
    print("Classical models:", CLASSICAL_MODEL_NAMES)
    print("Classical k values:", CLASSICAL_K_VALUES)

    for spec in task_specs:
        for sensor_combo in SENSOR_COMBINATIONS:
            print("\n" + "=" * 110)
            print(
                "CLASSICAL:",
                spec["task_name"],
                "|",
                sensor_combo,
            )
            print("=" * 110)

            summary, folds, preds, best = (
                run_classical_for_task_and_sensor(
                    spec,
                    sensor_combo,
                )
            )

            if not summary.empty:
                all_classical_summaries.append(summary)
            if not folds.empty:
                all_classical_folds.append(folds)
            if not best.empty:
                all_classical_best.append(best)

    if all_classical_summaries:
        combined_classical = pd.concat(
            all_classical_summaries,
            ignore_index=True,
        )
        combined_classical.to_csv(
            os.path.join(
                OUT_DIR,
                "combined_classical_summary_with_std.csv",
            ),
            index=False,
        )

    if all_classical_folds:
        combined_classical_folds = pd.concat(
            all_classical_folds,
            ignore_index=True,
        )
        combined_classical_folds.to_csv(
            os.path.join(
                OUT_DIR,
                "combined_classical_fold_metrics.csv",
            ),
            index=False,
        )

    if all_classical_best:
        combined_classical_best = pd.concat(
            all_classical_best,
            ignore_index=True,
        )
        combined_classical_best.to_csv(
            os.path.join(
                OUT_DIR,
                "combined_classical_best_per_condition_with_std.csv",
            ),
            index=False,
        )
        display(combined_classical_best.round(4))


Classical models: ['logreg_C1', 'linearSVC_C1']
Classical k values: [80, 200]

CLASSICAL: interaction_vs_noninteraction | OE
Classical | interaction_vs_noninteraction | OE | no_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OE | no_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OE | no_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OE | no_elapsed | linearSVC_C1 | k=200
Classical | interaction_vs_noninteraction | OE | with_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OE | with_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OE | with_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OE | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_interaction,recall_interaction,f1_interaction,support_interaction,precision_non_interaction,recall_non_interaction,f1_non_interaction,support_non_interaction,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.6357,0.6348,0.6361,0.6195,0.6914,0.6535,2275,0.6559,0.5807,0.6160,2304,interaction_vs_noninteraction,OE,no_elapsed,logreg_C1,200,457,4579,0.6392,0.0539,0.6047,0.0598,0.6185,0.0427,9
1,0.6988,0.6988,0.6989,0.6910,0.7125,0.7016,2275,0.7071,0.6853,0.6961,2304,interaction_vs_noninteraction,OE,with_elapsed,logreg_C1,200,458,4579,0.7029,0.1093,0.6831,0.1194,0.6951,0.0968,9



CLASSICAL: interaction_vs_noninteraction | OPTI
Classical | interaction_vs_noninteraction | OPTI | no_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI | no_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OPTI | no_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI | no_elapsed | linearSVC_C1 | k=200
Classical | interaction_vs_noninteraction | OPTI | with_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI | with_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OPTI | with_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_interaction,recall_interaction,f1_interaction,support_interaction,precision_non_interaction,recall_non_interaction,f1_non_interaction,support_non_interaction,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.7290,0.729,0.729,0.7268,0.7284,0.7276,2275,0.7312,0.7296,0.7304,2304,interaction_vs_noninteraction,OPTI,no_elapsed,linearSVC_C1,200,387,4579,0.7114,0.0989,0.6990,0.0943,0.7008,0.0899,9
1,0.7502,0.750,0.750,0.7598,0.7270,0.7430,2275,0.7415,0.7730,0.7569,2304,interaction_vs_noninteraction,OPTI,with_elapsed,linearSVC_C1,80,388,4579,0.7332,0.1122,0.7195,0.1161,0.7286,0.0929,9



CLASSICAL: interaction_vs_noninteraction | XSENS
Classical | interaction_vs_noninteraction | XSENS | no_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | XSENS | no_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | XSENS | no_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | XSENS | no_elapsed | linearSVC_C1 | k=200
Classical | interaction_vs_noninteraction | XSENS | with_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | XSENS | with_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | XSENS | with_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | XSENS | with_elapsed | linearSVC_C1 | k=200


...


CLASSICAL: interaction_vs_noninteraction | OPTI_XSENS
Classical | interaction_vs_noninteraction | OPTI_XSENS | no_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI_XSENS | no_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OPTI_XSENS | no_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI_XSENS | no_elapsed | linearSVC_C1 | k=200
Classical | interaction_vs_noninteraction | OPTI_XSENS | with_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI_XSENS | with_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OPTI_XSENS | with_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OPTI_XSENS | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_interaction,recall_interaction,f1_interaction,support_interaction,precision_non_interaction,recall_non_interaction,f1_non_interaction,support_non_interaction,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.7205,0.7205,0.7205,0.7156,0.7257,0.7206,2275,0.7254,0.7153,0.7203,2304,interaction_vs_noninteraction,OPTI_XSENS,no_elapsed,linearSVC_C1,80,1002,4579,0.7053,0.0957,0.7001,0.0923,0.7081,0.0965,9
1,0.7401,0.7399,0.7400,0.7506,0.7143,0.7320,2275,0.7307,0.7656,0.7478,2304,interaction_vs_noninteraction,OPTI_XSENS,with_elapsed,linearSVC_C1,80,1003,4579,0.7254,0.1140,0.7125,0.1178,0.7230,0.0961,9



CLASSICAL: interaction_vs_noninteraction | OE_OPTI_XSENS
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | no_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | no_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | no_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | no_elapsed | linearSVC_C1 | k=200
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | with_elapsed | logreg_C1 | k=80
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | with_elapsed | logreg_C1 | k=200
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | with_elapsed | linearSVC_C1 | k=80
Classical | interaction_vs_noninteraction | OE_OPTI_XSENS | with_elapsed | linearSVC_C1 | k=200


,accuracy,macro_f1,balanced_accuracy,precision_interaction,recall_interaction,f1_interaction,support_interaction,precision_non_interaction,recall_non_interaction,f1_non_interaction,support_non_interaction,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.7198,0.7198,0.7199,0.7121,0.7319,0.7219,2275,0.7278,0.7079,0.7177,2304,interaction_vs_noninteraction,OE_OPTI_XSENS,no_elapsed,linearSVC_C1,80,1459,4579,0.7033,0.0994,0.6978,0.0960,0.7064,0.0993,9
1,0.7360,0.7356,0.7358,0.7507,0.7015,0.7253,2275,0.7232,0.7700,0.7458,2304,interaction_vs_noninteraction,OE_OPTI_XSENS,with_elapsed,linearSVC_C1,80,1460,4579,0.7217,0.1137,0.7090,0.1171,0.7204,0.0954,9


,accuracy,macro_f1,balanced_accuracy,precision_interaction,recall_interaction,f1_interaction,support_interaction,precision_non_interaction,recall_non_interaction,f1_non_interaction,support_non_interaction,task,sensor_combo,time_condition,model,k,n_features,n_rows_evaluated,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.6357,0.6348,0.6361,0.6195,0.6914,0.6535,2275,0.6559,0.5807,0.6160,2304,interaction_vs_noninteraction,OE,no_elapsed,logreg_C1,200,457,4579,0.6392,0.0539,0.6047,0.0598,0.6185,0.0427,9
1,0.6988,0.6988,0.6989,0.6910,0.7125,0.7016,2275,0.7071,0.6853,0.6961,2304,interaction_vs_noninteraction,OE,with_elapsed,logreg_C1,200,458,4579,0.7029,0.1093,0.6831,0.1194,0.6951,0.0968,9
2,0.7290,0.7290,0.7290,0.7268,0.7284,0.7276,2275,0.7312,0.7296,0.7304,2304,interaction_vs_noninteraction,OPTI,no_elapsed,linearSVC_C1,200,387,4579,0.7114,0.0989,0.6990,0.0943,0.7008,0.0899,9
3,0.7502,0.7500,0.7500,0.7598,0.7270,0.7430,2275,0.7415,0.7730,0.7569,2304,interaction_vs_noninteraction,OPTI,with_elapsed,linearSVC_C1,80,388,4579,0.7332,0.1122,0.7195,0.1161,0.7286,0.0929,9
4,0.5165,0.5164,0.5166,0.5128,0.5354,0.5239,2275,0.5204,0.4978,0.5089,2304,interaction_vs_noninteraction,XSENS,no_elapsed,logreg_C1,80,615,4579,0.5264,0.1095,0.4644,0.1031,0.5076,0.0666,9
5,0.6672,0.6670,0.6671,0.6704,0.6492,0.6597,2275,0.6641,0.6849,0.6744,2304,interaction_vs_noninteraction,XSENS,with_elapsed,logreg_C1,80,616,4579,0.6562,0.0906,0.6156,0.1063,0.6388,0.0872,9
6,0.7250,0.7250,0.7251,0.7158,0.7407,0.7280,2275,0.7348,0.7096,0.7220,2304,interaction_vs_noninteraction,OE_OPTI,no_elapsed,linearSVC_C1,80,844,4579,0.7077,0.1006,0.7017,0.0972,0.7097,0.1000,9
7,0.7406,0.7404,0.7405,0.7463,0.7240,0.7349,2275,0.7352,0.7569,0.7459,2304,interaction_vs_noninteraction,OE_OPTI,with_elapsed,linearSVC_C1,80,845,4579,0.7259,0.1149,0.7116,0.1188,0.7219,0.0962,9
8,0.5532,0.5512,0.5536,0.5439,0.6233,0.5809,2275,0.5654,0.4839,0.5215,2304,interaction_vs_noninteraction,OE_XSENS,no_elapsed,logreg_C1,200,1072,4579,0.5634,0.0769,0.5137,0.0657,0.5452,0.0582,9
9,0.6624,0.6597,0.6630,0.6345,0.7560,0.6899,2275,0.7029,0.5699,0.6294,2304,interaction_vs_noninteraction,OE_XSENS,with_elapsed,logreg_C1,80,1073,4579,0.6669,0.0945,0.6307,0.1012,0.6451,0.0901,9


## Deep-learning comparison without elapsed time

LSTM, BiLSTM, GRU and Transformer are evaluated using the longest
context used in the original comparison: 90 seconds. Model selection
is performed separately for every task and sensor combination.

In [4]:
# ================================================================
# DL MODEL HELPERS
# ================================================================

class RNNClassifier(nn.Module):
    def __init__(self, input_dim, n_classes, rnn_type="lstm", hidden_dim=64, num_layers=1, dropout=0.25, bidirectional=False):
        super().__init__()
        rnn_cls = nn.LSTM if rnn_type == "lstm" else nn.GRU
        self.rnn = rnn_cls(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )
        out_dim = hidden_dim * (2 if bidirectional else 1)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Dropout(dropout),
            nn.Linear(out_dim, n_classes),
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        return self.head(last)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerClassifier(nn.Module):
    def __init__(self, input_dim, n_classes, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.25):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model=d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos(x)
        out = self.encoder(x)
        last = out[:, -1, :]
        return self.head(last)


ALL_MODEL_CONFIGS = [
    {"model_type": "lstm", "hidden_dim": 64, "num_layers": 1, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"model_type": "bilstm", "hidden_dim": 64, "num_layers": 1, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"model_type": "gru", "hidden_dim": 64, "num_layers": 1, "dropout": 0.25, "lr": 1e-3, "weight_decay": 1e-4},
    {"model_type": "transformer", "d_model": 64, "nhead": 4, "num_layers": 2, "dim_feedforward": 128, "dropout": 0.25, "lr": 5e-4, "weight_decay": 1e-4},
]

MODEL_CONFIGS = [m for m in ALL_MODEL_CONFIGS if m["model_type"] in RUN_DL_MODEL_TYPES]
print("DL model types:", [m["model_type"] for m in MODEL_CONFIGS])


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_model(model_cfg, input_dim, n_classes):
    mt = model_cfg["model_type"]
    if mt == "lstm":
        return RNNClassifier(input_dim, n_classes, rnn_type="lstm", hidden_dim=model_cfg["hidden_dim"], num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"], bidirectional=False)
    if mt == "bilstm":
        return RNNClassifier(input_dim, n_classes, rnn_type="lstm", hidden_dim=model_cfg["hidden_dim"], num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"], bidirectional=True)
    if mt == "gru":
        return RNNClassifier(input_dim, n_classes, rnn_type="gru", hidden_dim=model_cfg["hidden_dim"], num_layers=model_cfg["num_layers"], dropout=model_cfg["dropout"], bidirectional=False)
    if mt == "transformer":
        return TransformerClassifier(input_dim, n_classes, d_model=model_cfg["d_model"], nhead=model_cfg["nhead"], num_layers=model_cfg["num_layers"], dim_feedforward=model_cfg["dim_feedforward"], dropout=model_cfg["dropout"])
    raise ValueError(mt)


def make_loader(X, y, batch_size=64, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def make_sequences(X, y, groups, starts, seq_len):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y)
    groups = np.asarray(groups)
    starts = np.asarray(starts, dtype=float)

    Xs, ys, gs, sts = [], [], [], []

    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        idx = idx[np.argsort(starts[idx])]
        if len(idx) < seq_len:
            continue
        for end_pos in range(seq_len - 1, len(idx)):
            win_idx = idx[end_pos - seq_len + 1:end_pos + 1]
            Xs.append(X[win_idx])
            ys.append(y[idx[end_pos]])
            gs.append(g)
            sts.append(starts[idx[end_pos]])

    if len(Xs) == 0:
        return np.empty((0, seq_len, X.shape[1]), dtype=np.float32), np.array([]), np.array([]), np.array([])

    return np.stack(Xs).astype(np.float32), np.array(ys), np.array(gs), np.array(sts)


def torch_predict(model, X, batch_size=512):
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(DEVICE)
            logits = model(xb)
            preds.extend(logits.argmax(1).cpu().numpy())
    return np.array(preds)


def choose_validation_group(train_groups, y_all, groups_all):
    candidates = []
    for g in sorted(np.unique(train_groups)):
        mask = groups_all == g
        n_classes = len(np.unique(y_all[mask]))
        n_rows = int(mask.sum())
        candidates.append((n_classes >= 2, n_rows, g))
    candidates = sorted(candidates, reverse=True)
    return candidates[0][2]


DL model types: ['lstm', 'bilstm', 'gru', 'transformer']


In [ ]:
# ================================================================
# DL NO-ELAPSED LOGO RUNNER WITH FOLD MEAN ± SD
# ================================================================

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif

FAST_DL_TASKS_TO_RUN = None
FAST_DL_SENSOR_COMBOS_TO_RUN = None
FAST_DL_MODEL_TYPES = RUN_DL_MODEL_TYPES
FAST_DL_K = 120
FAST_DL_SEQ_CHOICE = "last"

FAST_DL_OUT_DIR = os.path.join(
    OUT_DIR,
    "DL_NO_ELAPSED_ALL_ARCHITECTURES",
)
os.makedirs(FAST_DL_OUT_DIR, exist_ok=True)


def choose_fast_seq_len(spec):
    seq_lens = list(spec["seq_lens"])
    if FAST_DL_SEQ_CHOICE == "last":
        return seq_lens[-1]
    if FAST_DL_SEQ_CHOICE == "first":
        return seq_lens[0]
    if FAST_DL_SEQ_CHOICE == "middle":
        return seq_lens[len(seq_lens) // 2]
    return int(FAST_DL_SEQ_CHOICE)


def get_fast_model_configs():
    return [
        cfg
        for cfg in MODEL_CONFIGS
        if cfg["model_type"] in FAST_DL_MODEL_TYPES
    ]


def torch_predict_fast(
    model,
    X,
    batch_size=FAST_PRED_BATCH_SIZE,
):
    model.eval()
    preds = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(
                X[i:i + batch_size],
                dtype=torch.float32,
            ).to(DEVICE)
            preds.extend(
                model(xb).argmax(1).cpu().numpy()
            )

    return np.asarray(preds)


def train_one_fast_dl_run(
    spec,
    sensor_combo,
    seq_len,
    k_features,
    model_cfg,
    seed,
):
    set_seed(seed)

    task_name = spec["task_name"]
    df = spec["df"]
    feature_list = spec["combo_features"][
        sensor_combo
    ].copy()
    label_order = spec["label_order"]
    n_classes = len(label_order)

    if not feature_list:
        return None, None, None

    label_to_id = {
        label: index
        for index, label in enumerate(label_order)
    }
    id_to_label = {
        index: label
        for label, index in label_to_id.items()
    }

    y_label = df[spec["target_col"]].astype(str).values
    y_all = np.asarray(
        [label_to_id[value] for value in y_label],
        dtype=int,
    )

    groups_all = df[spec["group_col"]].values
    starts_all = pd.to_numeric(
        df[spec["start_col"]],
        errors="coerce",
    ).values

    X_raw = (
        df[feature_list]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .values
    )

    logo = LeaveOneGroupOut()

    y_true_all = []
    y_pred_all = []
    fold_rows = []
    pred_rows = []

    for fold, (trval_idx, te_idx) in enumerate(
        logo.split(X_raw, y_all, groups_all),
        start=1,
    ):
        if (
            MAX_LOGO_FOLDS is not None
            and fold > MAX_LOGO_FOLDS
        ):
            break

        if len(np.unique(y_all[trval_idx])) < 2:
            continue

        test_group = groups_all[te_idx][0]

        train_groups = np.unique(groups_all[trval_idx])
        val_group = choose_validation_group(
            train_groups,
            y_all,
            groups_all,
        )

        val_mask = groups_all[trval_idx] == val_group
        val_idx = trval_idx[val_mask]
        tr_idx = trval_idx[~val_mask]

        if len(np.unique(y_all[tr_idx])) < 2:
            continue

        imputer = SimpleImputer(strategy="median")
        scaler = RobustScaler()

        Xtr = imputer.fit_transform(X_raw[tr_idx])
        Xval = imputer.transform(X_raw[val_idx])
        Xte = imputer.transform(X_raw[te_idx])

        Xtr = scaler.fit_transform(Xtr)
        Xval = scaler.transform(Xval)
        Xte = scaler.transform(Xte)

        actual_k = min(
            int(k_features),
            Xtr.shape[1],
        )

        selector = SelectKBest(
            f_classif,
            k=actual_k,
        )
        Xtr = selector.fit_transform(
            Xtr,
            y_all[tr_idx],
        )
        Xval = selector.transform(Xval)
        Xte = selector.transform(Xte)

        Xtr_seq, ytr_seq, _, _ = make_sequences(
            Xtr,
            y_all[tr_idx],
            groups_all[tr_idx],
            starts_all[tr_idx],
            seq_len,
        )
        Xval_seq, yval_seq, _, _ = make_sequences(
            Xval,
            y_all[val_idx],
            groups_all[val_idx],
            starts_all[val_idx],
            seq_len,
        )
        Xte_seq, yte_seq, gte_seq, ste_seq = (
            make_sequences(
                Xte,
                y_all[te_idx],
                groups_all[te_idx],
                starts_all[te_idx],
                seq_len,
            )
        )

        if (
            len(Xtr_seq) == 0
            or len(Xval_seq) == 0
            or len(Xte_seq) == 0
        ):
            continue

        model = build_model(
            model_cfg,
            input_dim=actual_k,
            n_classes=n_classes,
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=model_cfg["lr"],
            weight_decay=model_cfg["weight_decay"],
        )
        loss_fn = nn.CrossEntropyLoss()

        train_loader = make_loader(
            Xtr_seq,
            ytr_seq,
            batch_size=FAST_BATCH_SIZE,
            shuffle=True,
        )

        best_state = None
        best_val_macro = -np.inf
        patience_left = FAST_PATIENCE
        best_epoch = 0

        for epoch in range(
            1,
            FAST_MAX_EPOCHS + 1,
        ):
            model.train()

            for xb, yb in train_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                optimizer.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0,
                )
                optimizer.step()

            val_pred = torch_predict_fast(
                model,
                Xval_seq,
            )
            val_macro = f1_score(
                yval_seq,
                val_pred,
                labels=np.arange(n_classes),
                average="macro",
                zero_division=0,
            )

            if val_macro > best_val_macro:
                best_val_macro = val_macro
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value
                    in model.state_dict().items()
                }
                patience_left = FAST_PATIENCE
                best_epoch = epoch
            else:
                patience_left -= 1

            if patience_left <= 0:
                break

        if best_state is not None:
            model.load_state_dict(best_state)
            model.to(DEVICE)

        test_pred = torch_predict_fast(
            model,
            Xte_seq,
        )

        y_true_all.extend(yte_seq.tolist())
        y_pred_all.extend(test_pred.tolist())

        fold_metric = metric_dict(
            yte_seq,
            test_pred,
            list(range(n_classes)),
        )

        fold_rows.append({
            "task": task_name,
            "sensor_combo": sensor_combo,
            "time_condition": "no_elapsed",
            "model_type": model_cfg["model_type"],
            "seed": seed,
            "seq_len": seq_len,
            "context_seconds": (
                seq_len * spec["window_seconds"]
            ),
            "k_features": actual_k,
            "n_features_before_select": len(
                feature_list
            ),
            "fold": fold,
            "test_group": test_group,
            "n_sequences": len(Xte_seq),
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro,
            "accuracy": fold_metric["accuracy"],
            "macro_f1": fold_metric["macro_f1"],
            "balanced_accuracy": fold_metric[
                "balanced_accuracy"
            ],
        })

        if SAVE_FAST_DL_PREDICTIONS:
            for yt, yp, group, start in zip(
                yte_seq,
                test_pred,
                gte_seq,
                ste_seq,
            ):
                pred_rows.append({
                    "task": task_name,
                    "sensor_combo": sensor_combo,
                    "model_type": model_cfg["model_type"],
                    "seed": seed,
                    "seq_len": seq_len,
                    "group": group,
                    "window_start": start,
                    "y_true": id_to_label[int(yt)],
                    "y_pred": id_to_label[int(yp)],
                    "correct": int(yt) == int(yp),
                })

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not y_true_all:
        return None, None, None

    y_true_labels = np.asarray([
        id_to_label[int(value)]
        for value in y_true_all
    ])
    y_pred_labels = np.asarray([
        id_to_label[int(value)]
        for value in y_pred_all
    ])

    summary = metric_dict(
        y_true_labels,
        y_pred_labels,
        label_order,
    )
    summary.update({
        "task": task_name,
        "sensor_combo": sensor_combo,
        "time_condition": "no_elapsed",
        "model_type": model_cfg["model_type"],
        "seed": seed,
        "seq_len": seq_len,
        "context_seconds": (
            seq_len * spec["window_seconds"]
        ),
        "k_features": int(k_features),
        "n_features_before_select": len(
            feature_list
        ),
        "n_sequences_evaluated": len(
            y_true_labels
        ),
        "max_epochs": FAST_MAX_EPOCHS,
        "patience": FAST_PATIENCE,
    })

    return (
        summary,
        pd.DataFrame(fold_rows),
        pd.DataFrame(pred_rows),
    )


fast_model_configs = get_fast_model_configs()

planned = []
for spec in task_specs:
    seq_len = choose_fast_seq_len(spec)

    for sensor_combo in SENSOR_COMBINATIONS:
        if not spec["combo_features"][sensor_combo]:
            continue

        for seed in SEEDS:
            for model_cfg in fast_model_configs:
                planned.append({
                    "task": spec["task_name"],
                    "sensor_combo": sensor_combo,
                    "model": model_cfg["model_type"],
                    "seed": seed,
                    "seq_len": seq_len,
                    "k": min(
                        FAST_DL_K,
                        len(
                            spec["combo_features"][
                                sensor_combo
                            ]
                        ),
                    ),
                })

planned_df = pd.DataFrame(planned)
display(planned_df)
planned_df.to_csv(
    os.path.join(
        FAST_DL_OUT_DIR,
        "dl_plan.csv",
    ),
    index=False,
)

all_fast_summaries = []
all_fast_folds = []
all_fast_preds = []

if RUN_DL:
    for run_index, row in planned_df.iterrows():
        spec = next(
            item
            for item in task_specs
            if item["task_name"] == row["task"]
        )
        model_cfg = next(
            item
            for item in fast_model_configs
            if item["model_type"] == row["model"]
        )

        print("\n" + "=" * 110)
        print(
            f"DL {run_index + 1}/{len(planned_df)} | "
            f"{row['task']} | {row['sensor_combo']} | "
            f"{row['model']} | seed={row['seed']}"
        )
        print("=" * 110)

        summary, fold_df, pred_df = (
            train_one_fast_dl_run(
                spec=spec,
                sensor_combo=row["sensor_combo"],
                seq_len=int(row["seq_len"]),
                k_features=int(row["k"]),
                model_cfg=model_cfg,
                seed=int(row["seed"]),
            )
        )

        if summary is None:
            continue

        all_fast_summaries.append(summary)

        if not fold_df.empty:
            all_fast_folds.append(fold_df)

        if (
            SAVE_FAST_DL_PREDICTIONS
            and not pred_df.empty
        ):
            all_fast_preds.append(pred_df)

        pd.DataFrame(
            all_fast_summaries
        ).to_csv(
            os.path.join(
                FAST_DL_OUT_DIR,
                "combined_dl_no_elapsed_summary_partial.csv",
            ),
            index=False,
        )

if all_fast_summaries:
    combined_fast_dl = pd.DataFrame(
        all_fast_summaries
    )
    combined_fast_folds = pd.concat(
        all_fast_folds,
        ignore_index=True,
    )

    dl_group_keys = [
        "task",
        "sensor_combo",
        "time_condition",
        "model_type",
        "seed",
        "seq_len",
        "context_seconds",
        "k_features",
        "n_features_before_select",
    ]

    dl_fold_stats = (
        combined_fast_folds
        .groupby(
            dl_group_keys,
            as_index=False,
        )
        .agg(
            fold_accuracy_mean=("accuracy", "mean"),
            fold_accuracy_std=("accuracy", "std"),
            fold_macro_f1_mean=("macro_f1", "mean"),
            fold_macro_f1_std=("macro_f1", "std"),
            fold_balanced_accuracy_mean=(
                "balanced_accuracy", "mean"
            ),
            fold_balanced_accuracy_std=(
                "balanced_accuracy", "std"
            ),
            n_folds=("test_group", "nunique"),
        )
    )

    combined_fast_dl = combined_fast_dl.merge(
        dl_fold_stats,
        on=dl_group_keys,
        how="left",
        validate="one_to_one",
    )

    combined_fast_dl.to_csv(
        os.path.join(
            FAST_DL_OUT_DIR,
            "combined_dl_no_elapsed_summary_with_std.csv",
        ),
        index=False,
    )
    combined_fast_folds.to_csv(
        os.path.join(
            FAST_DL_OUT_DIR,
            "combined_dl_no_elapsed_fold_metrics.csv",
        ),
        index=False,
    )

    # Match the report: choose best by pooled macro-F1.
    combined_fast_dl_best = (
        combined_fast_dl
        .sort_values(
            [
                "task",
                "sensor_combo",
                "macro_f1",
                "accuracy",
            ],
            ascending=[
                True,
                True,
                False,
                False,
            ],
        )
        .groupby(
            ["task", "sensor_combo"],
            as_index=False,
        )
        .head(1)
        .reset_index(drop=True)
    )

    combined_fast_dl_best.to_csv(
        os.path.join(
            FAST_DL_OUT_DIR,
            "combined_dl_no_elapsed_best_per_task_sensor_with_std.csv",
        ),
        index=False,
    )

    display(combined_fast_dl_best.round(4))

    if len(SEEDS) > 1:
        seed_keys = [
            "task",
            "sensor_combo",
            "model_type",
            "seq_len",
            "context_seconds",
            "k_features",
            "n_features_before_select",
        ]
        dl_seed_stats = (
            combined_fast_dl
            .groupby(seed_keys, as_index=False)
            .agg(
                seed_accuracy_mean=("accuracy", "mean"),
                seed_accuracy_std=("accuracy", "std"),
                seed_macro_f1_mean=("macro_f1", "mean"),
                seed_macro_f1_std=("macro_f1", "std"),
                n_seeds=("seed", "nunique"),
            )
        )
        dl_seed_stats.to_csv(
            os.path.join(
                FAST_DL_OUT_DIR,
                "dl_seed_variability.csv",
            ),
            index=False,
        )
        display(dl_seed_stats.round(4))

    if (
        SAVE_FAST_DL_PREDICTIONS
        and all_fast_preds
    ):
        pd.concat(
            all_fast_preds,
            ignore_index=True,
        ).to_csv(
            os.path.join(
                FAST_DL_OUT_DIR,
                "combined_dl_no_elapsed_predictions.csv",
            ),
            index=False,
        )


,task,sensor_combo,model,seed,seq_len,k
0,interaction_vs_noninteraction,OE,lstm,42,18,120
1,interaction_vs_noninteraction,OE,bilstm,42,18,120
2,interaction_vs_noninteraction,OE,gru,42,18,120
3,interaction_vs_noninteraction,OE,transformer,42,18,120
4,interaction_vs_noninteraction,OPTI,lstm,42,18,120
5,interaction_vs_noninteraction,OPTI,bilstm,42,18,120
6,interaction_vs_noninteraction,OPTI,gru,42,18,120
7,interaction_vs_noninteraction,OPTI,transformer,42,18,120
8,interaction_vs_noninteraction,XSENS,lstm,42,18,120
9,interaction_vs_noninteraction,XSENS,bilstm,42,18,120



DL 1/28 | interaction_vs_noninteraction | OE | lstm | seed=42

DL 2/28 | interaction_vs_noninteraction | OE | bilstm | seed=42

DL 3/28 | interaction_vs_noninteraction | OE | gru | seed=42

DL 4/28 | interaction_vs_noninteraction | OE | transformer | seed=42

DL 5/28 | interaction_vs_noninteraction | OPTI | lstm | seed=42

DL 6/28 | interaction_vs_noninteraction | OPTI | bilstm | seed=42

DL 7/28 | interaction_vs_noninteraction | OPTI | gru | seed=42

DL 8/28 | interaction_vs_noninteraction | OPTI | transformer | seed=42

DL 9/28 | interaction_vs_noninteraction | XSENS | lstm | seed=42

DL 10/28 | interaction_vs_noninteraction | XSENS | bilstm | seed=42

DL 11/28 | interaction_vs_noninteraction | XSENS | gru | seed=42

DL 12/28 | interaction_vs_noninteraction | XSENS | transformer | seed=42

DL 13/28 | interaction_vs_noninteraction | OE_OPTI | lstm | seed=42

DL 14/28 | interaction_vs_noninteraction | OE_OPTI | bilstm | seed=42

DL 15/28 | interaction_vs_noninteraction | OE_OPTI | gru

,accuracy,macro_f1,balanced_accuracy,precision_interaction,recall_interaction,f1_interaction,support_interaction,precision_non_interaction,recall_non_interaction,f1_non_interaction,support_non_interaction,task,sensor_combo,time_condition,model_type,seed,seq_len,context_seconds,k_features,n_features_before_select,n_sequences_evaluated,max_epochs,patience,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds
0,0.5450,0.5443,0.5451,0.5504,0.5043,0.5263,2219,0.5403,0.5859,0.5622,2207,interaction_vs_noninteraction,OE,no_elapsed,bilstm,42,18,90.0,120,457,4426,25,4,0.5326,0.0674,0.4970,0.0798,0.5393,0.0515,9
1,0.7225,0.7225,0.7225,0.7170,0.7377,0.7272,2219,0.7284,0.7073,0.7177,2207,interaction_vs_noninteraction,OE_OPTI,no_elapsed,bilstm,42,18,90.0,120,844,4426,25,4,0.7221,0.0840,0.7135,0.0828,0.7215,0.0807,9
2,0.6597,0.6563,0.6595,0.6346,0.7575,0.6906,2219,0.6972,0.5614,0.6220,2207,interaction_vs_noninteraction,OE_OPTI_XSENS,no_elapsed,lstm,42,18,90.0,120,1459,4426,25,4,0.6532,0.1220,0.6322,0.1489,0.6663,0.0892,9
3,0.5228,0.5200,0.5230,0.5286,0.4452,0.4834,2219,0.5186,0.6008,0.5567,2207,interaction_vs_noninteraction,OE_XSENS,no_elapsed,transformer,42,18,90.0,120,1072,4426,25,4,0.5386,0.1199,0.4873,0.0983,0.5198,0.0802,9
4,0.7508,0.7508,0.7508,0.7516,0.7512,0.7514,2219,0.7500,0.7503,0.7502,2207,interaction_vs_noninteraction,OPTI,no_elapsed,transformer,42,18,90.0,120,387,4426,25,4,0.7320,0.1105,0.7115,0.1392,0.7367,0.1079,9
5,0.6629,0.6627,0.6628,0.6562,0.6881,0.6718,2219,0.6703,0.6375,0.6535,2207,interaction_vs_noninteraction,OPTI_XSENS,no_elapsed,transformer,42,18,90.0,120,1002,4426,25,4,0.6508,0.1301,0.6341,0.1410,0.6681,0.0928,9
6,0.5300,0.5151,0.5296,0.5233,0.7039,0.6003,2219,0.5441,0.3552,0.4298,2207,interaction_vs_noninteraction,XSENS,no_elapsed,bilstm,42,18,90.0,120,615,4426,25,4,0.5516,0.0852,0.4417,0.0668,0.5075,0.0399,9


## Final publication tables

The long table keeps all numeric columns. The wide table mirrors the
structure of the report, with one column for each of the three regimes.

In [ ]:
# ================================================================
# FINAL THREE-REGIME COMPARISON TABLES
# ================================================================

required_classical = {
    "combined_classical_best",
    "combined_classical_folds",
}
required_dl = {
    "combined_fast_dl_best",
    "combined_fast_folds",
}

missing = [
    name
    for name in required_classical | required_dl
    if name not in globals()
]
if missing:
    raise RuntimeError(
        "Run the classical and DL cells first. "
        f"Missing variables: {missing}"
    )

long_rows = []

for _, row in combined_classical_best.iterrows():
    regime = (
        "classical_no_elapsed"
        if row["time_condition"] == "no_elapsed"
        else "classical_with_elapsed"
    )

    long_rows.append({
        "task": row["task"],
        "sensor_combo": row["sensor_combo"],
        "regime": regime,
        "model_family": "classical",
        "best_model": row["model"],
        "selection": f"k={row['k']}",
        "n_features": row["n_features"],
        "pooled_accuracy": row["accuracy"],
        "pooled_macro_f1": row["macro_f1"],
        "pooled_balanced_accuracy": (
            row["balanced_accuracy"]
        ),
        "fold_accuracy_mean": (
            row["fold_accuracy_mean"]
        ),
        "fold_accuracy_std": (
            row["fold_accuracy_std"]
        ),
        "fold_macro_f1_mean": (
            row["fold_macro_f1_mean"]
        ),
        "fold_macro_f1_std": (
            row["fold_macro_f1_std"]
        ),
        "fold_balanced_accuracy_mean": (
            row["fold_balanced_accuracy_mean"]
        ),
        "fold_balanced_accuracy_std": (
            row["fold_balanced_accuracy_std"]
        ),
        "n_folds": row["n_folds"],
    })

for _, row in combined_fast_dl_best.iterrows():
    long_rows.append({
        "task": row["task"],
        "sensor_combo": row["sensor_combo"],
        "regime": "dl_no_elapsed",
        "model_family": "DL",
        "best_model": row["model_type"],
        "selection": (
            f"seq={int(row['seq_len'])}, "
            f"k={int(row['k_features'])}"
        ),
        "n_features": (
            row["n_features_before_select"]
        ),
        "pooled_accuracy": row["accuracy"],
        "pooled_macro_f1": row["macro_f1"],
        "pooled_balanced_accuracy": (
            row["balanced_accuracy"]
        ),
        "fold_accuracy_mean": (
            row["fold_accuracy_mean"]
        ),
        "fold_accuracy_std": (
            row["fold_accuracy_std"]
        ),
        "fold_macro_f1_mean": (
            row["fold_macro_f1_mean"]
        ),
        "fold_macro_f1_std": (
            row["fold_macro_f1_std"]
        ),
        "fold_balanced_accuracy_mean": (
            row["fold_balanced_accuracy_mean"]
        ),
        "fold_balanced_accuracy_std": (
            row["fold_balanced_accuracy_std"]
        ),
        "n_folds": row["n_folds"],
    })

publication_long = pd.DataFrame(long_rows)

publication_long["accuracy_mean_pm_std"] = (
    publication_long.apply(
        lambda row: (
            f"{row['fold_accuracy_mean']:.3f} "
            f"± {row['fold_accuracy_std']:.3f}"
        ),
        axis=1,
    )
)
publication_long["macro_f1_mean_pm_std"] = (
    publication_long.apply(
        lambda row: (
            f"{row['fold_macro_f1_mean']:.3f} "
            f"± {row['fold_macro_f1_std']:.3f}"
        ),
        axis=1,
    )
)
publication_long[
    "balanced_accuracy_mean_pm_std"
] = publication_long.apply(
    lambda row: (
        f"{row['fold_balanced_accuracy_mean']:.3f} "
        f"± {row['fold_balanced_accuracy_std']:.3f}"
    ),
    axis=1,
)

regime_order = [
    "classical_no_elapsed",
    "classical_with_elapsed",
    "dl_no_elapsed",
]
publication_long["regime"] = pd.Categorical(
    publication_long["regime"],
    categories=regime_order,
    ordered=True,
)
publication_long = publication_long.sort_values(
    ["task", "sensor_combo", "regime"]
).reset_index(drop=True)

publication_long.to_csv(
    os.path.join(
        OUT_DIR,
        "publication_three_regime_comparison_long.csv",
    ),
    index=False,
)

display_columns = [
    "task",
    "sensor_combo",
    "regime",
    "best_model",
    "selection",
    "pooled_accuracy",
    "pooled_macro_f1",
    "pooled_balanced_accuracy",
    "accuracy_mean_pm_std",
    "macro_f1_mean_pm_std",
    "balanced_accuracy_mean_pm_std",
]
display(
    publication_long[display_columns]
    .round(4)
)

# Compact wide table resembling the report.
compact = publication_long.copy()
compact["result_text"] = compact.apply(
    lambda row: (
        f"{row['best_model']} ({row['selection']}); "
        f"pooled A={row['pooled_accuracy']:.3f}, "
        f"M={row['pooled_macro_f1']:.3f}, "
        f"B={row['pooled_balanced_accuracy']:.3f}; "
        f"fold A={row['accuracy_mean_pm_std']}, "
        f"M={row['macro_f1_mean_pm_std']}"
    ),
    axis=1,
)

publication_wide = (
    compact.pivot(
        index=["task", "sensor_combo"],
        columns="regime",
        values="result_text",
    )
    .reset_index()
)

winner_rows = (
    publication_long.sort_values(
        [
            "task",
            "sensor_combo",
            "pooled_macro_f1",
            "pooled_accuracy",
        ],
        ascending=[
            True,
            True,
            False,
            False,
        ],
    )
    .groupby(
        ["task", "sensor_combo"],
        as_index=False,
    )
    .head(1)
    [[
        "task",
        "sensor_combo",
        "regime",
        "best_model",
        "pooled_macro_f1",
    ]]
    .rename(
        columns={
            "regime": "best_regime_by_pooled_macro_f1",
            "best_model": "best_model_overall",
            "pooled_macro_f1": (
                "best_pooled_macro_f1"
            ),
        }
    )
)

publication_wide = publication_wide.merge(
    winner_rows,
    on=["task", "sensor_combo"],
    how="left",
)

publication_wide.to_csv(
    os.path.join(
        OUT_DIR,
        "publication_three_regime_comparison_wide.csv",
    ),
    index=False,
)

display(publication_wide)

print("Saved all final tables to:")
print(OUT_DIR)


,task,sensor_combo,regime,best_model,selection,pooled_accuracy,pooled_macro_f1,pooled_balanced_accuracy,accuracy_mean_pm_std,macro_f1_mean_pm_std,balanced_accuracy_mean_pm_std
0,interaction_vs_noninteraction,OE,classical_no_elapsed,logreg_C1,k=200,0.6357,0.6348,0.6361,0.639 ± 0.054,0.605 ± 0.060,0.619 ± 0.043
1,interaction_vs_noninteraction,OE,classical_with_elapsed,logreg_C1,k=200,0.6988,0.6988,0.6989,0.703 ± 0.109,0.683 ± 0.119,0.695 ± 0.097
2,interaction_vs_noninteraction,OE,dl_no_elapsed,bilstm,"seq=18, k=120",0.5450,0.5443,0.5451,0.533 ± 0.067,0.497 ± 0.080,0.539 ± 0.052
3,interaction_vs_noninteraction,OE_OPTI,classical_no_elapsed,linearSVC_C1,k=80,0.7250,0.7250,0.7251,0.708 ± 0.101,0.702 ± 0.097,0.710 ± 0.100
4,interaction_vs_noninteraction,OE_OPTI,classical_with_elapsed,linearSVC_C1,k=80,0.7406,0.7404,0.7405,0.726 ± 0.115,0.712 ± 0.119,0.722 ± 0.096
5,interaction_vs_noninteraction,OE_OPTI,dl_no_elapsed,bilstm,"seq=18, k=120",0.7225,0.7225,0.7225,0.722 ± 0.084,0.714 ± 0.083,0.721 ± 0.081
6,interaction_vs_noninteraction,OE_OPTI_XSENS,classical_no_elapsed,linearSVC_C1,k=80,0.7198,0.7198,0.7199,0.703 ± 0.099,0.698 ± 0.096,0.706 ± 0.099
7,interaction_vs_noninteraction,OE_OPTI_XSENS,classical_with_elapsed,linearSVC_C1,k=80,0.7360,0.7356,0.7358,0.722 ± 0.114,0.709 ± 0.117,0.720 ± 0.095
8,interaction_vs_noninteraction,OE_OPTI_XSENS,dl_no_elapsed,lstm,"seq=18, k=120",0.6597,0.6563,0.6595,0.653 ± 0.122,0.632 ± 0.149,0.666 ± 0.089
9,interaction_vs_noninteraction,OE_XSENS,classical_no_elapsed,logreg_C1,k=200,0.5532,0.5512,0.5536,0.563 ± 0.077,0.514 ± 0.066,0.545 ± 0.058


,task,sensor_combo,classical_no_elapsed,classical_with_elapsed,dl_no_elapsed,best_regime_by_pooled_macro_f1,best_model_overall,best_pooled_macro_f1
0,interaction_vs_noninteraction,OE,"logreg_C1 (k=200); pooled A=0.636, M=0.635, B=0.636; fold A=0.639 ± 0.054, M=0.605 ± 0.060","logreg_C1 (k=200); pooled A=0.699, M=0.699, B=0.699; fold A=0.703 ± 0.109, M=0.683 ± 0.119","bilstm (seq=18, k=120); pooled A=0.545, M=0.544, B=0.545; fold A=0.533 ± 0.067, M=0.497 ± 0.080",classical_with_elapsed,logreg_C1,0.698817
1,interaction_vs_noninteraction,OE_OPTI,"linearSVC_C1 (k=80); pooled A=0.725, M=0.725, B=0.725; fold A=0.708 ± 0.101, M=0.702 ± 0.097","linearSVC_C1 (k=80); pooled A=0.741, M=0.740, B=0.740; fold A=0.726 ± 0.115, M=0.712 ± 0.119","bilstm (seq=18, k=120); pooled A=0.723, M=0.722, B=0.723; fold A=0.722 ± 0.084, M=0.714 ± 0.083",classical_with_elapsed,linearSVC_C1,0.740438
2,interaction_vs_noninteraction,OE_OPTI_XSENS,"linearSVC_C1 (k=80); pooled A=0.720, M=0.720, B=0.720; fold A=0.703 ± 0.099, M=0.698 ± 0.096","linearSVC_C1 (k=80); pooled A=0.736, M=0.736, B=0.736; fold A=0.722 ± 0.114, M=0.709 ± 0.117","lstm (seq=18, k=120); pooled A=0.660, M=0.656, B=0.659; fold A=0.653 ± 0.122, M=0.632 ± 0.149",classical_with_elapsed,linearSVC_C1,0.735569
3,interaction_vs_noninteraction,OE_XSENS,"logreg_C1 (k=200); pooled A=0.553, M=0.551, B=0.554; fold A=0.563 ± 0.077, M=0.514 ± 0.066","logreg_C1 (k=80); pooled A=0.662, M=0.660, B=0.663; fold A=0.667 ± 0.095, M=0.631 ± 0.101","transformer (seq=18, k=120); pooled A=0.523, M=0.520, B=0.523; fold A=0.539 ± 0.120, M=0.487 ± 0.098",classical_with_elapsed,logreg_C1,0.659683
4,interaction_vs_noninteraction,OPTI,"linearSVC_C1 (k=200); pooled A=0.729, M=0.729, B=0.729; fold A=0.711 ± 0.099, M=0.699 ± 0.094","linearSVC_C1 (k=80); pooled A=0.750, M=0.750, B=0.750; fold A=0.733 ± 0.112, M=0.720 ± 0.116","transformer (seq=18, k=120); pooled A=0.751, M=0.751, B=0.751; fold A=0.732 ± 0.110, M=0.711 ± 0.139",dl_no_elapsed,transformer,0.750789
5,interaction_vs_noninteraction,OPTI_XSENS,"linearSVC_C1 (k=80); pooled A=0.720, M=0.720, B=0.720; fold A=0.705 ± 0.096, M=0.700 ± 0.092","linearSVC_C1 (k=80); pooled A=0.740, M=0.740, B=0.740; fold A=0.725 ± 0.114, M=0.713 ± 0.118","transformer (seq=18, k=120); pooled A=0.663, M=0.663, B=0.663; fold A=0.651 ± 0.130, M=0.634 ± 0.141",classical_with_elapsed,linearSVC_C1,0.739878
6,interaction_vs_noninteraction,XSENS,"logreg_C1 (k=80); pooled A=0.516, M=0.516, B=0.517; fold A=0.526 ± 0.109, M=0.464 ± 0.103","logreg_C1 (k=80); pooled A=0.667, M=0.667, B=0.667; fold A=0.656 ± 0.091, M=0.616 ± 0.106","bilstm (seq=18, k=120); pooled A=0.530, M=0.515, B=0.530; fold A=0.552 ± 0.085, M=0.442 ± 0.067",classical_with_elapsed,logreg_C1,0.667014


Saved all final tables to:
/content/drive/MyDrive/thesis/data/PUBLICATION_TASK1_FULL_COMPARISON


In [5]:
# ================================================================
# TASK 1 — EXACT OPTIMIZED TRANSFORMER REPRODUCTION
# Run AFTER the existing "DL MODEL HELPERS" cell.
# ================================================================

import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

# Exact historical configuration
DATA_PATH = (
    "/content/drive/MyDrive/thesis/data/"
    "INTERACTION_BINARY_5S_SPECIALIZED_OE/"
    "binary_5s_specialized_oe_merged_all_features.csv"
)

OUT_DIR_0806 = (
    "/content/drive/MyDrive/thesis/data/"
    "INTERACTION_BINARY_5S_SPECIALIZED_OE/"
    "PUBLICATION_TASK1_OPTIMIZED_0806"
)

os.makedirs(OUT_DIR_0806, exist_ok=True)

SEED = 42
SEQ_LEN = 18               # 18 × 5 seconds = 90 seconds
K = 120
MAX_EPOCHS = 80
PATIENCE = 12
BATCH_SIZE = 64

# Reuse definitions from the existing DL MODEL HELPERS cell
for required in [
    "DEVICE",
    "TransformerClassifier",
    "make_loader",
    "make_sequences",
]:
    if required not in globals():
        raise RuntimeError(
            "Run the existing 'DL MODEL HELPERS' cell first. "
            f"Missing: {required}"
        )

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}\n"
        "Check that Google Drive is mounted."
    )

df = pd.read_csv(DATA_PATH)

# ================================================================
# Exact historical OPTI2_RELATIVE_ONLY feature definition
# ================================================================

TOKENS = [
    "dist",
    "spread",
    "area",
    "speed",
    "active_speed",
    "pair",
    "nearest",
    "farthest",
    "triangle",
]


def is_relative_opti2(column):
    name = str(column)
    lower = name.lower()

    return (
        (
            name.startswith("opti2_")
            or name.startswith("opti2__")
        )
        and any(token in lower for token in TOKENS)
    )


features = []

for column in df.columns:
    if not is_relative_opti2(column):
        continue

    if column == "elapsed_min":
        continue

    values = pd.to_numeric(
        df[column],
        errors="coerce",
    ).to_numpy(dtype=float)

    if np.isfinite(values).sum() < 20:
        continue

    if np.nanstd(values) < 1e-12:
        continue

    features.append(column)

print("Dataset shape:", df.shape)
print("OPTI2_RELATIVE_ONLY features:", len(features))

# This guarantees that the same feature definition was recovered
if len(features) != 211:
    raise RuntimeError(
        "The exact historical feature set was not recovered. "
        f"Expected 211 features, found {len(features)}."
    )

# ================================================================
# Data arrays
# ================================================================

label_to_id = {
    "non_interaction": 0,
    "interaction": 1,
}

y = (
    df["binary_label"]
    .map(label_to_id)
    .to_numpy(dtype=int)
)

groups = df["group"].to_numpy()

starts = pd.to_numeric(
    df["window_start"],
    errors="raise",
).to_numpy(dtype=float)

X_raw = (
    df[features]
    .apply(pd.to_numeric, errors="coerce")
    .to_numpy(dtype=float)
)

display(df["binary_label"].value_counts())
display(pd.crosstab(df["group"], df["binary_label"]))

# ================================================================
# Exact historical training helpers
# ================================================================

def set_exact_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def impute_from_training(X_train, X_test):
    medians = np.nanmedian(X_train, axis=0)

    medians = np.where(
        np.isfinite(medians),
        medians,
        0.0,
    )

    X_train = np.where(
        np.isfinite(X_train),
        X_train,
        medians,
    )

    X_test = np.where(
        np.isfinite(X_test),
        X_test,
        medians,
    )

    return X_train, X_test


def predict_loader_exact(model, loader):
    model.eval()

    true_values = []
    predictions = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)

            pred = (
                model(xb)
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            predictions.extend(pred.tolist())
            true_values.extend(yb.numpy().tolist())

    return (
        np.asarray(true_values),
        np.asarray(predictions),
    )


def train_exact_fold(
    X_train_sequences,
    y_train_sequences,
    group_train_sequences,
    X_test_sequences,
    y_test_sequences,
):
    # Exact old behavior: reset seed 42 for every LOGO fold
    set_exact_seed(SEED)

    # Exact old validation rule:
    # use the largest remaining group ID
    validation_group = sorted(
        np.unique(group_train_sequences)
    )[-1]

    fit_mask = (
        group_train_sequences != validation_group
    )

    validation_mask = (
        group_train_sequences == validation_group
    )

    # Historical fallback, normally not triggered
    if (
        fit_mask.sum() < 50
        or validation_mask.sum() < 50
    ):
        rng = np.random.default_rng(SEED)

        indices = np.arange(
            len(y_train_sequences)
        )

        rng.shuffle(indices)

        n_validation = max(
            50,
            int(0.15 * len(indices)),
        )

        fit_mask = np.zeros(
            len(indices),
            dtype=bool,
        )

        validation_mask = np.zeros(
            len(indices),
            dtype=bool,
        )

        validation_mask[
            indices[:n_validation]
        ] = True

        fit_mask[
            indices[n_validation:]
        ] = True

    X_fit = X_train_sequences[fit_mask]
    y_fit = y_train_sequences[fit_mask]

    X_validation = (
        X_train_sequences[validation_mask]
    )

    y_validation = (
        y_train_sequences[validation_mask]
    )

    train_loader = make_loader(
        X_fit,
        y_fit,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    validation_loader = make_loader(
        X_validation,
        y_validation,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_loader = make_loader(
        X_test_sequences,
        y_test_sequences,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    # Exact historical Transformer architecture
    model = TransformerClassifier(
        input_dim=X_train_sequences.shape[-1],
        n_classes=2,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.25,
    ).to(DEVICE)

    # Exact historical class-weighted loss
    class_counts = np.bincount(
        y_fit,
        minlength=2,
    ).astype(float)

    class_weights = (
        class_counts.sum()
        / (
            2.0
            * np.maximum(class_counts, 1.0)
        )
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    loss_function = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=5e-4,
        weight_decay=1e-4,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=4,
            min_lr=1e-5,
        )
    )

    best_state = None
    best_validation_macro_f1 = -np.inf
    best_epoch = 0
    bad_epochs = 0

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)

            loss = loss_function(
                logits,
                yb,
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

        validation_true, validation_pred = (
            predict_loader_exact(
                model,
                validation_loader,
            )
        )

        validation_macro_f1 = f1_score(
            validation_true,
            validation_pred,
            average="macro",
            zero_division=0,
        )

        scheduler.step(
            validation_macro_f1
        )

        if (
            validation_macro_f1
            > best_validation_macro_f1
        ):
            best_validation_macro_f1 = (
                validation_macro_f1
            )

            best_epoch = epoch
            bad_epochs = 0

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            break

    if best_state is None:
        raise RuntimeError(
            "No valid model state was saved."
        )

    model.load_state_dict(
        best_state
    )

    test_true, test_pred = (
        predict_loader_exact(
            model,
            test_loader,
        )
    )

    return {
        "y_true": test_true,
        "y_pred": test_pred,
        "validation_group": validation_group,
        "best_epoch": best_epoch,
        "best_validation_macro_f1": (
            best_validation_macro_f1
        ),
    }


# ================================================================
# Exact leave-one-group-out evaluation
# ================================================================

logo = LeaveOneGroupOut()

all_true = []
all_pred = []
fold_rows = []

for fold, (
    train_indices,
    test_indices,
) in enumerate(
    logo.split(
        X_raw,
        y,
        groups,
    ),
    start=1,
):
    test_group = groups[
        test_indices
    ][0]

    X_train, X_test = (
        impute_from_training(
            X_raw[train_indices],
            X_raw[test_indices],
        )
    )

    # Exact historical RobustScaler
    scaler = RobustScaler()

    X_train = scaler.fit_transform(
        X_train
    )

    X_test = scaler.transform(
        X_test
    )

    # Exact historical in-fold selection
    actual_k = min(
        K,
        X_train.shape[1] - 1,
    )

    selector = SelectKBest(
        score_func=f_classif,
        k=actual_k,
    )

    X_train = selector.fit_transform(
        X_train,
        y[train_indices],
    )

    X_test = selector.transform(
        X_test
    )

    (
        X_train_sequences,
        y_train_sequences,
        group_train_sequences,
        _,
    ) = make_sequences(
        X_train,
        y[train_indices],
        groups[train_indices],
        starts[train_indices],
        SEQ_LEN,
    )

    (
        X_test_sequences,
        y_test_sequences,
        _,
        _,
    ) = make_sequences(
        X_test,
        y[test_indices],
        groups[test_indices],
        starts[test_indices],
        SEQ_LEN,
    )

    print(
        f"Fold {fold}/9 | "
        f"test group={test_group} | "
        f"train sequences={len(X_train_sequences)} | "
        f"test sequences={len(X_test_sequences)}"
    )

    result = train_exact_fold(
        X_train_sequences,
        y_train_sequences,
        group_train_sequences,
        X_test_sequences,
        y_test_sequences,
    )

    y_true_fold = result["y_true"]
    y_pred_fold = result["y_pred"]

    fold_accuracy = accuracy_score(
        y_true_fold,
        y_pred_fold,
    )

    fold_macro_f1 = f1_score(
        y_true_fold,
        y_pred_fold,
        average="macro",
        zero_division=0,
    )

    fold_balanced_accuracy = (
        balanced_accuracy_score(
            y_true_fold,
            y_pred_fold,
        )
    )

    fold_rows.append({
        "fold": fold,
        "test_group": test_group,
        "validation_group": (
            result["validation_group"]
        ),
        "n_test_sequences": len(
            y_true_fold
        ),
        "accuracy": fold_accuracy,
        "macro_f1": fold_macro_f1,
        "balanced_accuracy": (
            fold_balanced_accuracy
        ),
        "best_epoch": (
            result["best_epoch"]
        ),
        "best_validation_macro_f1": (
            result[
                "best_validation_macro_f1"
            ]
        ),
    })

    all_true.extend(
        y_true_fold.tolist()
    )

    all_pred.extend(
        y_pred_fold.tolist()
    )

    print(
        f"  accuracy={fold_accuracy:.4f} | "
        f"macro-F1={fold_macro_f1:.4f} | "
        f"balanced={fold_balanced_accuracy:.4f} | "
        f"best_epoch={result['best_epoch']}"
    )

all_true = np.asarray(all_true)
all_pred = np.asarray(all_pred)

fold_metrics_0806 = pd.DataFrame(
    fold_rows
)

# Exact historical number of evaluated sequences
if len(all_true) != 4426:
    raise RuntimeError(
        "Historical sequence count mismatch. "
        f"Expected 4426, found {len(all_true)}."
    )

# ================================================================
# Pooled results and fold mean ± SD
# ================================================================

summary_0806 = pd.DataFrame([{
    "run_name": (
        "OPTI2_REL_seq18_k120_"
        "transformer_seed42"
    ),
    "feature_set": (
        "OPTI2_RELATIVE_ONLY"
    ),
    "model": "Transformer",
    "seed": SEED,
    "sequence_length": SEQ_LEN,
    "context_seconds": 90,
    "available_features": len(
        features
    ),
    "selected_features": actual_k,
    "n_sequences_evaluated": len(
        all_true
    ),
    "pooled_accuracy": accuracy_score(
        all_true,
        all_pred,
    ),
    "pooled_macro_f1": f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
    "pooled_balanced_accuracy": (
        balanced_accuracy_score(
            all_true,
            all_pred,
        )
    ),
    "fold_accuracy_mean": (
        fold_metrics_0806[
            "accuracy"
        ].mean()
    ),
    "fold_accuracy_std": (
        fold_metrics_0806[
            "accuracy"
        ].std(ddof=1)
    ),
    "fold_macro_f1_mean": (
        fold_metrics_0806[
            "macro_f1"
        ].mean()
    ),
    "fold_macro_f1_std": (
        fold_metrics_0806[
            "macro_f1"
        ].std(ddof=1)
    ),
    "fold_balanced_accuracy_mean": (
        fold_metrics_0806[
            "balanced_accuracy"
        ].mean()
    ),
    "fold_balanced_accuracy_std": (
        fold_metrics_0806[
            "balanced_accuracy"
        ].std(ddof=1)
    ),
    "n_folds": (
        fold_metrics_0806[
            "test_group"
        ].nunique()
    ),
}])

summary_0806[
    "accuracy_mean_pm_std"
] = summary_0806.apply(
    lambda row: (
        f"{row['fold_accuracy_mean']:.3f} "
        f"± {row['fold_accuracy_std']:.3f}"
    ),
    axis=1,
)

summary_0806[
    "macro_f1_mean_pm_std"
] = summary_0806.apply(
    lambda row: (
        f"{row['fold_macro_f1_mean']:.3f} "
        f"± {row['fold_macro_f1_std']:.3f}"
    ),
    axis=1,
)

summary_0806[
    "balanced_accuracy_mean_pm_std"
] = summary_0806.apply(
    lambda row: (
        f"{row['fold_balanced_accuracy_mean']:.3f} "
        f"± {row['fold_balanced_accuracy_std']:.3f}"
    ),
    axis=1,
)

print(
    "\nOPTIMIZED HISTORICAL TASK 1 RESULT"
)

display(
    summary_0806.round(4)
)

print(
    "\nFold-level results"
)

display(
    fold_metrics_0806.round(4)
)

# ================================================================
# Save outputs
# ================================================================

summary_0806.to_csv(
    os.path.join(
        OUT_DIR_0806,
        "task1_optimized_0806_summary_with_std.csv",
    ),
    index=False,
)

fold_metrics_0806.to_csv(
    os.path.join(
        OUT_DIR_0806,
        "task1_optimized_0806_fold_metrics.csv",
    ),
    index=False,
)

pd.DataFrame({
    "feature_name": features
}).to_csv(
    os.path.join(
        OUT_DIR_0806,
        "task1_optimized_0806_exact_features.csv",
    ),
    index=False,
)

print("\nHistorical target:")
print("pooled accuracy      ≈ 0.8064")
print("pooled macro-F1      ≈ 0.8062")
print("pooled balanced acc. ≈ 0.8065")

Dataset shape: (4579, 1515)
OPTI2_RELATIVE_ONLY features: 211


,count
binary_label,
non_interaction,2304
interaction,2275


binary_label,interaction,non_interaction
group,,
1,213,153
2,299,267
3,205,330
5,391,401
6,152,122
7,332,247
8,244,220
9,361,352
10,78,212


Fold 1/9 | test group=1 | train sequences=4077 | test sequences=349
  accuracy=0.6361 | macro-F1=0.6234 | balanced=0.6796 | best_epoch=1
Fold 2/9 | test group=2 | train sequences=3877 | test sequences=549
  accuracy=0.7978 | macro-F1=0.7977 | balanced=0.8023 | best_epoch=15
Fold 3/9 | test group=3 | train sequences=3908 | test sequences=518
  accuracy=0.8707 | macro-F1=0.8576 | balanced=0.8466 | best_epoch=8
Fold 4/9 | test group=5 | train sequences=3651 | test sequences=775
  accuracy=0.8903 | macro-F1=0.8902 | balanced=0.8901 | best_epoch=6
Fold 5/9 | test group=6 | train sequences=4169 | test sequences=257
  accuracy=0.6848 | macro-F1=0.6640 | balanced=0.6614 | best_epoch=5
Fold 6/9 | test group=7 | train sequences=3864 | test sequences=562
  accuracy=0.8167 | macro-F1=0.8128 | balanced=0.8134 | best_epoch=1
Fold 7/9 | test group=8 | train sequences=3979 | test sequences=447
  accuracy=0.7808 | macro-F1=0.7808 | balanced=0.7826 | best_epoch=1
Fold 8/9 | test group=9 | train sequence

,run_name,feature_set,model,seed,sequence_length,context_seconds,available_features,selected_features,n_sequences_evaluated,pooled_accuracy,pooled_macro_f1,pooled_balanced_accuracy,fold_accuracy_mean,fold_accuracy_std,fold_macro_f1_mean,fold_macro_f1_std,fold_balanced_accuracy_mean,fold_balanced_accuracy_std,n_folds,accuracy_mean_pm_std,macro_f1_mean_pm_std,balanced_accuracy_mean_pm_std
0,OPTI2_REL_seq18_k120_transformer_seed42,OPTI2_RELATIVE_ONLY,Transformer,42,18,90,211,120,4426,0.8064,0.8062,0.8065,0.7889,0.0815,0.7815,0.0856,0.7927,0.0765,9,0.789 ± 0.082,0.781 ± 0.086,0.793 ± 0.076



Fold-level results


,fold,test_group,validation_group,n_test_sequences,accuracy,macro_f1,balanced_accuracy,best_epoch,best_validation_macro_f1
0,1,1,10,349,0.6361,0.6234,0.6796,1,0.7844
1,2,2,10,549,0.7978,0.7977,0.8023,15,0.7995
2,3,3,10,518,0.8707,0.8576,0.8466,8,0.7940
3,4,5,10,775,0.8903,0.8902,0.8901,6,0.7982
4,5,6,10,257,0.6848,0.6640,0.6614,5,0.7886
5,6,7,10,562,0.8167,0.8128,0.8134,1,0.7800
6,7,8,10,447,0.7808,0.7808,0.7826,1,0.7800
7,8,9,10,696,0.8060,0.8057,0.8057,1,0.7800
8,9,10,9,273,0.8168,0.8010,0.8526,5,0.8046



Historical target:
pooled accuracy      ≈ 0.8064
pooled macro-F1      ≈ 0.8062
pooled balanced acc. ≈ 0.8065


In [6]:
# =====================================================================
# TASK 1 OPTIMIZED — NAIVE-5 RERUN
# Hucre 15'ten SONRA calistir. Ayni feature seti, ayni mimari,
# ayni seed, ayni fold mantigi; sadece gruplar degisiyor.
# =====================================================================
import numpy as np, pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

NAIVE_GROUPS = {2, 3, 5, 6, 10}

for required in ["df", "features", "train_exact_fold", "impute_from_training",
                 "make_sequences", "SEED", "SEQ_LEN", "K"]:
    if required not in globals():
        raise RuntimeError(f"Once hucre 15'i calistir. Eksik: {required}")

print(f"feature sayisi (tam veriden turetilmis): {len(features)}")   # 211 olmali

# ---- satirlari naive gruplara indir; feature TANIMI degismiyor ----
def _gid(g):
    d = "".join(ch for ch in str(g) if ch.isdigit())
    return int(d) if d else -1

mask_n = df["group"].map(_gid).isin(NAIVE_GROUPS).to_numpy()
dfn = df.loc[mask_n].copy()
print(f"satir: {len(df)} -> {len(dfn)}   gruplar: {sorted(set(dfn['group'].map(_gid)))}")

label_to_id = {"non_interaction": 0, "interaction": 1}
yn      = dfn["binary_label"].map(label_to_id).to_numpy(dtype=int)
groupsn = dfn["group"].to_numpy()
startsn = pd.to_numeric(dfn["window_start"], errors="raise").to_numpy(dtype=float)
Xn_raw  = dfn[features].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
display(pd.crosstab(dfn["group"], dfn["binary_label"]))

# ---- LOGO: hucre 15'teki dongunun birebir aynisi ----
logo = LeaveOneGroupOut()
all_true_n, all_pred_n, fold_rows_n = [], [], []

for fold, (tr, te) in enumerate(logo.split(Xn_raw, yn, groupsn), start=1):
    test_group = groupsn[te][0]
    X_train, X_test = impute_from_training(Xn_raw[tr], Xn_raw[te])

    scaler  = RobustScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    actual_k = min(K, X_train.shape[1] - 1)
    selector = SelectKBest(score_func=f_classif, k=actual_k)
    X_train  = selector.fit_transform(X_train, yn[tr])
    X_test   = selector.transform(X_test)

    Xtr_seq, ytr_seq, gtr_seq, _ = make_sequences(
        X_train, yn[tr], groupsn[tr], startsn[tr], SEQ_LEN)
    Xte_seq, yte_seq, _, _ = make_sequences(
        X_test,  yn[te], groupsn[te], startsn[te], SEQ_LEN)

    if len(Xtr_seq) == 0 or len(Xte_seq) == 0:
        print(f"Fold {fold}: bos sekans, atlaniyor"); continue

    print(f"Fold {fold}/5 | test grubu={test_group} | "
          f"train seq={len(Xtr_seq)} | test seq={len(Xte_seq)}")

    res = train_exact_fold(Xtr_seq, ytr_seq, gtr_seq, Xte_seq, yte_seq)

    yt, yp = res["y_true"], res["y_pred"]
    all_true_n.extend(yt.tolist()); all_pred_n.extend(yp.tolist())
    fold_rows_n.append({
        "fold": fold, "test_group": _gid(test_group),
        "validation_group": _gid(res["validation_group"]),
        "n_test_sequences": len(yt),
        "accuracy": accuracy_score(yt, yp),
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(yt, yp),
        "best_epoch": res["best_epoch"],
        "best_validation_macro_f1": res["best_validation_macro_f1"],
    })

fold_n = pd.DataFrame(fold_rows_n).sort_values("test_group").reset_index(drop=True)
at, ap = np.array(all_true_n), np.array(all_pred_n)

pooled_acc = accuracy_score(at, ap)
pooled_mf1 = f1_score(at, ap, average="macro", zero_division=0)
pooled_bal = balanced_accuracy_score(at, ap)

v  = fold_n["macro_f1"].to_numpy(dtype=float)
m  = v.mean(); sd = v.std(ddof=1)
t4 = 2.776                      # 5 fold
lo, hi = m - t4*sd/np.sqrt(len(v)), m + t4*sd/np.sqrt(len(v))

print("\n" + "=" * 78)
print("TASK 1 OPTIMIZED (OPTI2_RELATIVE_ONLY, Transformer, seq=18, k=120) — NAIVE 5")
print("=" * 78)
print(fold_n[["test_group", "n_test_sequences", "accuracy",
              "macro_f1", "balanced_accuracy"]].to_string(index=False))
print(f"\npooled  A={pooled_acc:.4f}  M={pooled_mf1:.4f}  B={pooled_bal:.4f}")
print(f"fold    mean={m:.4f} +/- {sd:.4f}   95% CI=[{max(lo,0):.4f}, {min(hi,1):.4f}]  (n={len(v)})")
print("\nfull-9 referans:  A=0.8064  M=0.8062  B=0.8065  fold 0.781+/-0.086 [0.716, 0.847]")
print(f"dM = {pooled_mf1 - 0.8062:+.4f}")

fold_n.to_csv(f"{OUT_DIR_0806}/task1_optimized_0806_fold_metrics_NAIVE5.csv", index=False)
print(f"\nkaydedildi -> {OUT_DIR_0806}/task1_optimized_0806_fold_metrics_NAIVE5.csv")
fold_n

feature sayisi (tam veriden turetilmis): 211
satir: 4579 -> 2457   gruplar: [2, 3, 5, 6, 10]


binary_label,interaction,non_interaction
group,,
2,299,267
3,205,330
5,391,401
6,152,122
10,78,212


Fold 1/5 | test grubu=2 | train seq=1823 | test seq=549
Fold 2/5 | test grubu=3 | train seq=1854 | test seq=518
Fold 3/5 | test grubu=5 | train seq=1597 | test seq=775
Fold 4/5 | test grubu=6 | train seq=2115 | test seq=257
Fold 5/5 | test grubu=10 | train seq=2099 | test seq=273

TASK 1 OPTIMIZED (OPTI2_RELATIVE_ONLY, Transformer, seq=18, k=120) — NAIVE 5
 test_group  n_test_sequences  accuracy  macro_f1  balanced_accuracy
          2               549  0.743169  0.743115           0.747826
          3               518  0.627413  0.627011           0.657218
          5               775  0.749677  0.743209           0.748305
          6               257  0.579767  0.577920           0.612343
         10               273  0.772894  0.753265           0.798718

pooled  A=0.7057  M=0.7057  B=0.7086
fold    mean=0.6889 +/- 0.0809   95% CI=[0.5885, 0.7893]  (n=5)

full-9 referans:  A=0.8064  M=0.8062  B=0.8065  fold 0.781+/-0.086 [0.716, 0.847]
dM = -0.1005

kaydedildi -> /content/drive

,fold,test_group,validation_group,n_test_sequences,accuracy,macro_f1,balanced_accuracy,best_epoch,best_validation_macro_f1
0,1,2,10,549,0.743169,0.743115,0.747826,1,0.787579
1,2,3,10,518,0.627413,0.627011,0.657218,1,0.784357
2,3,5,10,775,0.749677,0.743209,0.748305,17,0.788700
3,4,6,10,257,0.579767,0.577920,0.612343,1,0.771185
4,5,10,6,273,0.772894,0.753265,0.798718,4,0.656417
